# 10. COCO Captions pilot (single run)

Goal: validate the VLM pipeline on COCO before a full sweep.

- Architecture: frozen SigLIP-Base/16@224 -> linear connector -> large GPT decoder
- Variant: baseline only
- Seed: 123
- Dataset: COCO captions (`train` for training, `validation` for holdout eval), with image-level disjointness enforcement
- Reuses existing training/checkpoint/W&B/Drive logic from `experiments/vlm_attnres_flickr30k.py`

This notebook is designed to:
1. Inspect/print data loading + split discipline + step-count rationale + CU estimate
2. Wait for your explicit confirmation
3. Launch training only when confirmed
4. Run corpus-level caption eval on the final checkpoint (CIDEr, BLEU-4, METEOR, ROUGE-L)

In [1]:
import os
import sys
import time
import math
import types
import argparse
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Iterable, Sequence

import torch
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset

# optional metric deps (eval cell later)
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pycocoevalcap"], check=True)

REPO_URL = "https://github.com/AtinChing/AttnResGPT-next-scale.git"
REPO_TARGET = Path("/content/AttnResGPT-next-scale")
DRIVE_ARTIFACT_FOLDER = "AttnResGPT-next-scale-artifacts"


def is_repo_root(path: Path) -> bool:
    return (path / "experiments" / "vlm_attnres_flickr30k.py").exists() and (path / "src").exists()


def discover_repo() -> Path:
    for candidate in [REPO_TARGET, Path.cwd(), Path.cwd().parent]:
        if is_repo_root(candidate):
            return candidate.resolve()
    if not REPO_TARGET.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_TARGET)], check=True)
    return REPO_TARGET.resolve()


REPO = discover_repo()
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print("repo:", REPO)

repo: /content/AttnResGPT-next-scale


In [2]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_ARTIFACT_FOLDER
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("drive root:", DRIVE_ROOT)

Mounted at /content/drive
drive root: /content/drive/MyDrive/AttnResGPT-next-scale-artifacts


## COCO data loader with image-level split discipline

This mirrors the Flickr loader style but enforces disjoint train/val image sets using official COCO splits (`train` vs `validation`).

In [3]:
from src.data.tokenizer import TokenizerWrapper
from src.utils.runtime import manual_seed_generator, seed_worker


@dataclass
class CaptionRecord:
    row_index: int
    caption: str
    image_id: str


def _extract_captions(payload: Any) -> Iterable[str]:
    if isinstance(payload, str):
        text = payload.strip()
        if text:
            yield text
    elif isinstance(payload, list):
        for item in payload:
            yield from _extract_captions(item)
    elif isinstance(payload, dict):
        for key in ("caption", "text", "raw", "sentence"):
            if key in payload:
                yield from _extract_captions(payload[key])


def _row_captions(row: dict[str, Any]) -> list[str]:
    for key in ("caption", "captions", "sentences", "annotations", "labels"):
        if key in row:
            caps = list(_extract_captions(row[key]))
            if caps:
                return caps
    return []


def _row_image(row: dict[str, Any]) -> Any:
    for key in ("image", "img", "jpg"):
        if key in row:
            return row[key]
    raise KeyError("Could not find image field in COCO row")


def _row_image_id(row: dict[str, Any], fallback_idx: int) -> str:
    for key in ("image_id", "img_id", "id", "cocoid"):
        if key in row and row[key] is not None:
            return str(row[key])
    image = row.get("image")
    if isinstance(image, dict):
        for key in ("path", "filename", "file_name"):
            if key in image and image[key]:
                return str(image[key])
    return f"row-{fallback_idx}"


class CaptionExampleDataset(Dataset):
    def __init__(self, dataset: Any, examples: Sequence[CaptionRecord]) -> None:
        self.dataset = dataset
        self.examples = list(examples)

    def __len__(self) -> int:
        return len(self.examples)

    def __getitem__(self, index: int) -> dict[str, Any]:
        record = self.examples[index]
        row = self.dataset[int(record.row_index)]
        return {
            "image": _row_image(row),
            "caption": record.caption,
            "image_id": record.image_id,
        }


class CocoCaptionCollator:
    def __init__(self, *, processor: Any, tokenizer: TokenizerWrapper, max_text_tokens: int) -> None:
        self.processor = processor
        self.tokenizer = tokenizer
        self.max_text_tokens = max_text_tokens
        backend = tokenizer.backend
        self.pad_token_id = int(backend.pad_token_id if backend.pad_token_id is not None else 0)
        bos = backend.bos_token_id if backend.bos_token_id is not None else backend.eos_token_id
        self.bos_token_id = int(bos if bos is not None else self.pad_token_id)

    def __call__(self, examples: Sequence[dict[str, Any]]) -> dict[str, torch.Tensor]:
        images = [example["image"] for example in examples]
        captions = [example["caption"] for example in examples]
        pixel_values = self.processor(images=images, return_tensors="pt")["pixel_values"]

        input_rows: list[list[int]] = []
        target_rows: list[list[int]] = []
        for caption in captions:
            token_ids = self.tokenizer.encode(caption)[: self.max_text_tokens]
            if not token_ids:
                token_ids = [self.bos_token_id]
            decoder_input = [self.bos_token_id, *token_ids[:-1]]
            input_rows.append(decoder_input)
            target_rows.append(token_ids)

        max_len = max(len(row) for row in input_rows)
        input_ids = torch.full((len(input_rows), max_len), self.pad_token_id, dtype=torch.long)
        targets = torch.full((len(input_rows), max_len), -100, dtype=torch.long)
        text_mask = torch.zeros((len(input_rows), max_len), dtype=torch.bool)

        for i, (decoder_input, decoder_target) in enumerate(zip(input_rows, target_rows)):
            input_ids[i, : len(decoder_input)] = torch.tensor(decoder_input, dtype=torch.long)
            targets[i, : len(decoder_target)] = torch.tensor(decoder_target, dtype=torch.long)
            text_mask[i, : len(decoder_input)] = True

        return {
            "pixel_values": pixel_values,
            "input_ids": input_ids,
            "targets": targets,
            "text_mask": text_mask,
        }


def _collect_examples(dataset: Any, *, max_examples: int | None = None) -> list[CaptionRecord]:
    examples: list[CaptionRecord] = []
    for row_index, row in enumerate(dataset):
        image_id = _row_image_id(row, row_index)
        for caption in _row_captions(row):
            examples.append(CaptionRecord(row_index=row_index, caption=caption, image_id=image_id))
            if max_examples is not None and len(examples) >= max_examples:
                return examples
    return examples


def build_coco_caption_dataloaders(
    *,
    dataset_name: str,
    train_split: str,
    val_split: str,
    processor: Any,
    tokenizer: TokenizerWrapper,
    max_train_examples: int,
    max_val_examples: int,
    max_text_tokens: int,
    batch_size: int,
    seed: int,
    num_workers: int,
) -> tuple[DataLoader, DataLoader, dict[str, int]]:
    train_dataset = load_dataset(dataset_name, split=train_split)
    val_dataset = load_dataset(dataset_name, split=val_split)

    if hasattr(train_dataset, "shuffle"):
        train_dataset = train_dataset.shuffle(seed=seed)

    train_examples = _collect_examples(train_dataset, max_examples=max_train_examples)
    val_examples = _collect_examples(val_dataset, max_examples=max_val_examples)

    train_image_ids = {ex.image_id for ex in train_examples}
    val_examples = [ex for ex in val_examples if ex.image_id not in train_image_ids]

    collator = CocoCaptionCollator(processor=processor, tokenizer=tokenizer, max_text_tokens=max_text_tokens)

    train_loader_kwargs: dict[str, Any] = {
        "batch_size": batch_size,
        "shuffle": True,
        "collate_fn": collator,
        "num_workers": num_workers,
        "generator": manual_seed_generator(seed),
    }
    if num_workers > 0:
        train_loader_kwargs["persistent_workers"] = True
        train_loader_kwargs["worker_init_fn"] = seed_worker

    train_loader = DataLoader(CaptionExampleDataset(train_dataset, train_examples), **train_loader_kwargs)
    val_loader = DataLoader(
        CaptionExampleDataset(val_dataset, val_examples),
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collator,
        num_workers=num_workers,
        persistent_workers=num_workers > 0,
    )

    stats = {
        "train_caption_examples": len(train_examples),
        "val_caption_examples": len(val_examples),
        "train_unique_images": len({ex.image_id for ex in train_examples}),
        "val_unique_images": len({ex.image_id for ex in val_examples}),
    }
    return train_loader, val_loader, stats

## Pilot config + rationale (printed before launch)

In [4]:
# ---- pilot knobs ----
DATASET_NAME = "yerevann/coco-karpathy"
TRAIN_SPLIT = "train"
VAL_SPLIT = "validation"

SEED = 123
DECODER_ARCH = "baseline"
BATCH_SIZE = 8
MAX_TEXT_TOKENS = 128
NUM_WORKERS = 4

# Keep train cap high enough for COCO scale, but finite for pilot robustness.
MAX_TRAIN_EXAMPLES = 600_000
MAX_VAL_EXAMPLES = 100_000

# Substantially longer than Flickr (6750): pilot target for COCO scale.
TARGET_STEPS = 24_000
CHECKPOINT_INTERVAL = 1_000

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 1_000

RUN_NAME = f"vlm_{DECODER_ARCH}_coco_b{BATCH_SIZE}_seed{SEED}_pilot"

# rough CU estimate (T4 typical range; adjust if runtime reports differ)
EST_SECONDS_PER_STEP = 0.9
EST_HOURS = TARGET_STEPS * EST_SECONDS_PER_STEP / 3600
EST_CU_PER_HOUR_LOW = 1.8
EST_CU_PER_HOUR_HIGH = 2.8
EST_CU_LOW = EST_HOURS * EST_CU_PER_HOUR_LOW
EST_CU_HIGH = EST_HOURS * EST_CU_PER_HOUR_HIGH

print("=== COCO pilot pre-launch summary ===")
print("Dataset loading:")
print(f"  dataset_name={DATASET_NAME}")
print(f"  train_split={TRAIN_SPLIT} | val_split={VAL_SPLIT}")
print("  captions extracted from row fields (caption/captions/sentences/annotations)")
print("  val examples filtered to drop any image_id seen in train (image-level disjointness)")
print()
print("Step-count reasoning:")
print("  Flickr run used 6750 steps on a much smaller caption set.")
print(f"  COCO pilot uses TARGET_STEPS={TARGET_STEPS} (~{TARGET_STEPS/6750:.1f}x Flickr) to be substantially longer")
print("  while still feasible as a single Colab pilot before sweeping.")
print()
print("Estimated Colab CU cost (single run):")
print(f"  runtime ~= {EST_HOURS:.1f} hours (assuming {EST_SECONDS_PER_STEP:.1f}s/step)")
print(f"  cost ~= {EST_CU_LOW:.1f} to {EST_CU_HIGH:.1f} compute units")

=== COCO pilot pre-launch summary ===
Dataset loading:
  dataset_name=yerevann/coco-karpathy
  train_split=train | val_split=validation
  captions extracted from row fields (caption/captions/sentences/annotations)
  val examples filtered to drop any image_id seen in train (image-level disjointness)

Step-count reasoning:
  Flickr run used 6750 steps on a much smaller caption set.
  COCO pilot uses TARGET_STEPS=24000 (~3.6x Flickr) to be substantially longer
  while still feasible as a single Colab pilot before sweeping.

Estimated Colab CU cost (single run):
  runtime ~= 6.0 hours (assuming 0.9s/step)
  cost ~= 10.8 to 16.8 compute units


## Build trainer by reusing existing training/checkpoint logic

In [5]:
import experiments.vlm_attnres_flickr30k as train_mod
from src.data.tokenizer import build_tokenizer
from src.models.vlm_attnres import SiglipAttnResCaptioner


def _choose_batch_size(_args) -> int:
    return int(_args.batch_size) if int(_args.batch_size) > 0 else 1


def build_coco_loaders_for_train_mod(
    *,
    dataset_name: str,
    split: str,
    processor: Any,
    tokenizer: TokenizerWrapper,
    max_examples: int,
    max_text_tokens: int,
    batch_size: int,
    seed: int,
    num_workers: int,
):
    # split arg is ignored here; we use explicit TRAIN_SPLIT/VAL_SPLIT constants.
    train_loader, val_loader, stats = build_coco_caption_dataloaders(
        dataset_name=dataset_name,
        train_split=TRAIN_SPLIT,
        val_split=VAL_SPLIT,
        processor=processor,
        tokenizer=tokenizer,
        max_train_examples=max_examples,
        max_val_examples=MAX_VAL_EXAMPLES,
        max_text_tokens=max_text_tokens,
        batch_size=batch_size,
        seed=seed,
        num_workers=num_workers,
    )
    print("COCO loader stats:", stats)
    return train_loader, val_loader


# monkey-patch loader hook used by existing trainer
train_mod.build_flickr30k_dataloaders = build_coco_loaders_for_train_mod
print("Patched train_mod.build_flickr30k_dataloaders -> COCO loader")

Patched train_mod.build_flickr30k_dataloaders -> COCO loader


## Confirmation gate (must be True to launch training)

In [6]:
CONFIRM_LAUNCH = False

args = argparse.Namespace(
    project="attnres-next-scale",
    entity=None,
    run_name=RUN_NAME,
    vision_model="google/siglip-base-patch16-224",
    dataset_name=DATASET_NAME,
    dataset_split=TRAIN_SPLIT,
    tokenizer_name="gpt2",
    decoder_config="configs/large_ctx512_3000.yaml",
    decoder_architecture=DECODER_ARCH,
    decoder_backend=f"gpt_{DECODER_ARCH}_large",
    device="auto",
    batch_size=BATCH_SIZE,
    max_examples=MAX_TRAIN_EXAMPLES,
    max_text_tokens=MAX_TEXT_TOKENS,
    num_workers=NUM_WORKERS,
    epochs=1,  # we train by explicit step target; this will be overridden below
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
    checkpoint_interval=CHECKPOINT_INTERVAL,
    eval_max_batches=None,
    alpha_eval_max_batches=None,
    plateau_patience=10_000,  # effectively disable early-stop for this pilot
    resume_from=None,
    wandb_resume="never",
    mixed_precision=True,
    amp_dtype="float16",
    seed=SEED,
)

print("Ready to launch with run_name:", args.run_name)
print("Set CONFIRM_LAUNCH=True, then run next cell to start training.")

Ready to launch with run_name: vlm_baseline_coco_b8_seed123_pilot
Set CONFIRM_LAUNCH=True, then run next cell to start training.


In [7]:
# Estimate steps/epoch from dataset cardinality so we can choose epochs for TARGET_STEPS.
train_ds = load_dataset(DATASET_NAME, split=TRAIN_SPLIT)
val_ds = load_dataset(DATASET_NAME, split=VAL_SPLIT)

train_examples_preview = _collect_examples(train_ds, max_examples=MAX_TRAIN_EXAMPLES)
train_caption_count = len(train_examples_preview)
steps_per_epoch = max(1, math.ceil(train_caption_count / BATCH_SIZE))
args.epochs = max(1, math.ceil(TARGET_STEPS / steps_per_epoch))
planned_total_steps = args.epochs * steps_per_epoch

print("train caption examples (capped):", train_caption_count)
print("steps_per_epoch:", steps_per_epoch)
print("epochs set to:", args.epochs)
print("planned_total_steps:", planned_total_steps)
print("target_steps:", TARGET_STEPS)

# Free preview memory before launch.
del train_examples_preview

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/246 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/17.0M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/82783 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating restval split:   0%|          | 0/30504 [00:00<?, ? examples/s]

train caption examples (capped): 414113
steps_per_epoch: 51765
epochs set to: 1
planned_total_steps: 51765
target_steps: 24000


In [8]:
if not CONFIRM_LAUNCH:
    raise RuntimeError("Set CONFIRM_LAUNCH=True in previous cell after reviewing rationale.")

launch_start = time.time()
train_mod.run_vlm(args)
print(f"Training finished in {(time.time() - launch_start)/3600:.2f} hours")

RuntimeError: Set CONFIRM_LAUNCH=True in previous cell after reviewing rationale.

In [ ]:
# Mirror checkpoints + outputs to Drive after training.
import shutil

local_ckpt_dir = REPO / "checkpoints" / RUN_NAME
local_out_dir = REPO / "outputs" / RUN_NAME
drive_ckpt_dir = DRIVE_ROOT / "checkpoints" / RUN_NAME
drive_out_dir = DRIVE_ROOT / "outputs" / RUN_NAME

for src, dst in [(local_ckpt_dir, drive_ckpt_dir), (local_out_dir, drive_out_dir)]:
    if src.exists():
        dst.mkdir(parents=True, exist_ok=True)
        for path in src.glob("*"):
            if path.is_file():
                shutil.copy2(path, dst / path.name)
        print("synced:", src, "->", dst)
    else:
        print("missing local dir:", src)

## Final-checkpoint caption eval (corpus-level metrics)

Runs greedy generation (40-token cap) on COCO val and scores with corpus-level CIDEr/BLEU-4/METEOR/ROUGE-L.

In [ ]:
import json
from collections import defaultdict

from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer

from src.utils.config import load_config
from src.models.vlm_attnres import SiglipAttnResCaptioner
from src.data.tokenizer import build_tokenizer


def latest_checkpoint(path: Path) -> Path:
    ckpts = sorted(path.glob("step_*.pt"))
    if not ckpts:
        raise FileNotFoundError(f"No checkpoints found in {path}")
    return ckpts[-1]


def build_coco_eval_rows(dataset_name: str, split: str, seed: int, max_rows: int = 0) -> list[dict[str, Any]]:
    ds = load_dataset(dataset_name, split=split)
    if hasattr(ds, "shuffle"):
        ds = ds.shuffle(seed=seed)
    rows = []
    for idx, row in enumerate(ds):
        caps = _row_captions(row)
        if not caps:
            continue
        rows.append({"row_index": idx, "image": _row_image(row), "references": caps})
        if max_rows and len(rows) >= max_rows:
            break
    return rows


def generate_captions(model: SiglipAttnResCaptioner, tokenizer, rows: list[dict[str, Any]], device: torch.device, max_gen_tokens: int = 40) -> list[dict]:
    model.eval()
    bos = tokenizer.backend.bos_token_id
    if bos is None:
        bos = tokenizer.backend.eos_token_id
    if bos is None:
        bos = tokenizer.backend.pad_token_id or 0
    eos = tokenizer.backend.eos_token_id
    if eos is None:
        eos = tokenizer.backend.pad_token_id or 0

    records = []
    with torch.no_grad():
        for row in rows:
            pixel = model.processor(images=[row["image"]], return_tensors="pt")["pixel_values"].to(device)
            vision_hidden = model.encode_vision(pixel)
            input_ids = torch.tensor([[int(bos)]], dtype=torch.long, device=device)
            for _ in range(max_gen_tokens):
                out = model(vision_hidden=vision_hidden, input_ids=input_ids, return_aux=False)
                nxt = out["logits"][:, -1, :].argmax(dim=-1)
                input_ids = torch.cat([input_ids, nxt.unsqueeze(1)], dim=1)
                if int(nxt.item()) == int(eos):
                    break
            tokens = []
            for t in input_ids[0, 1:].tolist():
                if int(t) == int(eos):
                    break
                tokens.append(int(t))
            pred = tokenizer.backend.decode(tokens).strip()
            records.append({
                "image_id": int(row["row_index"]),
                "prediction": pred,
                "references": list(row["references"]),
            })
    return records


def score_corpus(records: list[dict]) -> dict[str, float]:
    gts = {r["image_id"]: [{"caption": c} for c in r["references"] if c.strip()] for r in records}
    res = {r["image_id"]: [{"caption": r["prediction"] if r["prediction"].strip() else "."}] for r in records}
    valid = [k for k, v in gts.items() if v]
    gts = {k: gts[k] for k in valid}
    res = {k: res[k] for k in valid}

    tok = PTBTokenizer()
    gts_t, res_t = tok.tokenize(gts), tok.tokenize(res)

    out = {}
    out["CIDEr"], _ = Cider().compute_score(gts_t, res_t)
    bleu, _ = Bleu(4).compute_score(gts_t, res_t)
    out["Bleu_4"] = float(bleu[3])
    out["ROUGE_L"], _ = Rouge().compute_score(gts_t, res_t)
    out["METEOR"], _ = Meteor().compute_score(gts_t, res_t)
    out["CIDEr"] = float(out["CIDEr"])
    out["ROUGE_L"] = float(out["ROUGE_L"])
    out["METEOR"] = float(out["METEOR"])
    return out


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt = latest_checkpoint(REPO / "checkpoints" / RUN_NAME)
print("evaluating checkpoint:", ckpt)

cfg = load_config(REPO / "configs/large_ctx512_3000.yaml", overrides=["model.architecture=baseline", "data.block_size=512", "model.max_seq_len=512"])
tok = build_tokenizer("gpt2")
cfg.model.vocab_size = tok.vocab_size
model = SiglipAttnResCaptioner(vision_model_name="google/siglip-base-patch16-224", decoder_config=cfg.model).to(device)
state = torch.load(ckpt, map_location=device)
model.load_state_dict(state["model"], strict=True)

val_rows = build_coco_eval_rows(DATASET_NAME, VAL_SPLIT, SEED, max_rows=2000)
print("eval rows:", len(val_rows))

records = generate_captions(model, tok, val_rows, device=device, max_gen_tokens=40)
metrics = score_corpus(records)
print("COCO pilot final metrics:")
for key in ["CIDEr", "Bleu_4", "METEOR", "ROUGE_L"]:
    print(f"  {key}: {metrics[key]:.4f}")

pilot_eval_path = REPO / "outputs" / RUN_NAME / "coco_pilot_eval_metrics.json"
pilot_eval_path.write_text(json.dumps(metrics, indent=2))
print("wrote", pilot_eval_path)